In [ ]:
import os
import sqlite3
import sys

sys.path.append(os.path.abspath(os.path.join("..")))
from typing import cast

from IPython.display import Image, display
from langchain_core.messages import HumanMessage, RemoveMessage, SystemMessage
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, MessagesState, StateGraph

from utils.model import Models


#### __Goals__
- Langgraph support memory checkpointers which could be internal memory checkpointers that cannot persist indefinitely, for this 
- There are external DB like sqlite and postgresql which can save checkpoints indefinitely 

In [ ]:
model = Models()
llm = model.create_groq_model()

In [ ]:
# INFO Create the path for db file
db_path = "./state_db/chatbot.db"


In [ ]:
# INFO Create the connection instance
conn = sqlite3.connect(db_path, check_same_thread=False)


In [ ]:
# INFO Connected the sqlite connection instance with SqliteSaver which the graph can access
memory = SqliteSaver(conn)

In [ ]:
class State(MessagesState, total=False):
    summary: str

In [ ]:
# INFO Same as the last one
def call_llm(state: State):
    summary = state.get("summary")
    if summary:
        sys_msg = f"here is the summary of the conversation so far {summary}"
        messages = [SystemMessage(sys_msg)] + state["messages"]
    else:
        messages = state["messages"]
    response = llm.invoke(messages)
    return {"messages": response}

In [ ]:
def summarize_conversation(state: State):
    summary = state.get("summary")
    if summary:
        input_msg = (
            f"Here is the summary of the conversation so far {summary}"
            "Take the new messages and the summary into consideration"
        )
        messages = [HumanMessage(content=input_msg)] + state["messages"]
    else:
        input_msg = "Create the summary of the conversation so far"
        messages = [HumanMessage(input_msg)] + state["messages"]
    response = llm.invoke(messages)
    delete_messages = [
        RemoveMessage(id=cast(str, m.id)) for m in state["messages"][:-2]
    ]
    return {"messages": delete_messages, "summary": response.content}

In [ ]:
def should_summarize(state: State):
    messages = state.get("messages")
    if len(messages) > 6:
        return "summarize_conversation"
    else:
        return END

In [ ]:
builder = StateGraph(State)
builder.add_node("call_llm", call_llm)
builder.add_node("summarize_conversation", summarize_conversation)
builder.add_edge(START, "call_llm")
builder.add_conditional_edges(
    "call_llm", should_summarize, ["summarize_conversation", END]
)
builder.add_edge("summarize_conversation", END)
workflow = builder.compile(checkpointer=memory)
display(Image(workflow.get_graph().draw_mermaid_png()))


In [16]:
input_config: RunnableConfig = {"configurable": {"thread_id": "1"}}

In [ ]:
input_msg = HumanMessage(content="hi")
output = workflow.invoke({"messages": [input_msg]}, config=input_config)
for m in output["messages"]:
    m.pretty_print()

In [ ]:
input_msg = HumanMessage("I am Jason")
output = workflow.invoke({"messages": [input_msg]}, config=input_config)
for m in output["messages"]:
    m.pretty_print()

In [18]:
graph_state = workflow.get_state(input_config)
graph_state.values

{'messages': [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='b6ecc3a4-e69a-46f3-9f63-5b72607f47c7'),
  AIMessage(content='Hello! How can I help you today?', additional_kwargs={'reasoning_content': 'We need to respond as ChatGPT. The user says "hi". We should respond politely. Use friendly greeting.'}, response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 72, 'total_tokens': 114, 'completion_time': 0.08785079, 'completion_tokens_details': {'reasoning_tokens': 24}, 'prompt_time': 0.00267333, 'prompt_tokens_details': None, 'queue_time': 0.213169597, 'total_time': 0.09052412}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_a655021f7d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a154be-db79-7373-95d3-be98ed800573-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 42, 'total_tokens': 114, 'output_token_details': {